# Milestone 2: four-arrangement quartet training

Train fresh seed-0 weights on `learned`, `transfer_02`, `transfer_04`, `transfer_06`.
Reserve `transfer_01`, `transfer_03`, `transfer_05`, `transfer_07` for final transfer evaluation.
Complete quartets and all shape/color identities are retained. Each population has
2,304 images / 6,912 questions. Keep the existing model, R=2, float32 and AdamW.

**8,640 updates / 276,480 presentations / 40 full passes**. Monitor training fit only;
no transfer monitoring, early stopping, resume, extension or best-checkpoint selection.
Fourfold updates versus the single-arrangement run; this does not isolate diversity.

Enable GPU and internet; attach the completed quartet-fit and quartet-transfer archives.
Set a committed `REPO_REF` after pushing. No project validation/test inference.
See `docs/milestones/milestone2_multi_arrangement.md`.


In [ ]:
REPO_URL = "https://github.com/Krailon/multi-modal-loop-llm.git"
REPO_REF = "milestone2"  # Commit SHA preferred; must contain this implementation.
REPO_DIR = "/kaggle/working/multi-modal-loop-multi-arrangement"
RUN_ROOT = "/kaggle/working/milestone2_multi_arrangement"
FIT_SOURCE = (
    "/kaggle/input/REPLACE_WITH_YOUR_ARTIFACT_PATH/milestone2_quartet_fit_artifacts.zip"
)
TRANSFER_SOURCE = (
    "/kaggle/input/REPLACE_WITH_YOUR_ARTIFACT_PATH/milestone2_quartet_transfer_artifacts.zip"
)


## Checkout and install

Run all cells in order. The resolved revision is recorded with the artifacts.


In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR).resolve()
RUN_ROOT = Path(RUN_ROOT).resolve()
if REPO_DIR == RUN_ROOT or REPO_DIR.is_relative_to(RUN_ROOT) or RUN_ROOT.is_relative_to(REPO_DIR):
    raise ValueError("Repository and artifacts must use separate directories")
if not REPO_REF or REPO_REF.startswith("-"):
    raise ValueError("Set REPO_REF to a branch, tag, or commit")


def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()


if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
else:
    if git("remote", "get-url", "origin") != REPO_URL:
        raise ValueError("Existing checkout belongs to a different repository")
    if git("status", "--porcelain"):
        raise ValueError("Existing checkout has local changes; use a clean committed checkout")
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    if git("rev-parse", "HEAD") != git("rev-parse", "FETCH_HEAD"):
        raise ValueError(
            "Existing checkout has a different revision. Set REPO_REF to its recorded commit "
            "to resume, or use a new REPO_DIR for a new run."
        )

resolved_revision = git("rev-parse", "HEAD")
identity = (str(REPO_DIR), resolved_revision)
if globals().get("_notebook_code_identity", identity) != identity:
    raise RuntimeError(
        "The kernel previously loaded another revision; restart it before proceeding"
    )
_notebook_code_identity = identity
print("Resolved code revision:", resolved_revision)
torch_version = importlib.metadata.version("torch")
# No torch extra, requirements.txt, or accelerator replacement.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)
if importlib.metadata.version("torch") != torch_version:
    raise RuntimeError("PyTorch changed during installation; inspect the environment")
# Editable-install .pth files are processed at interpreter startup. Make this
# checkout importable immediately in the running notebook kernel too.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
# Training/evaluation run in subprocesses using this kernel's Python interpreter.

## Audit references and prepare the fixed split

Accept ZIPs or extracted directories. Audit corpus, checkpoint identity and saved transfer
predictions; reference weights never initialize training. Assign alternating arrangements
by the established order, preserving balance and separation. Previous transfer scores are
known, so this is exploratory development rather than an untouched-test experiment.


In [ ]:
import multimodal_loop.eval.kaggle_multi_arrangement as helpers
from multimodal_loop.eval.kaggle_multi_arrangement import (
    archive_multi_arrangement,
    prepare_multi_arrangement,
    run_multi_arrangement,
)

if not Path(helpers.__file__).resolve().is_relative_to(REPO_DIR / "src"):
    raise RuntimeError("Another package checkout is cached; restart the kernel")
run = prepare_multi_arrangement(REPO_DIR, RUN_ROOT, FIT_SOURCE, TRANSFER_SOURCE)


## Train and evaluate the final checkpoint

Run 40 passes, with training-fit monitoring at step zero and every 216 updates.
At the final checkpoint, evaluate all eight arrangements and compare transfer on exactly
the same four reserved arrangements with saved single-arrangement-model predictions.


In [ ]:
report = run_multi_arrangement(run)


## Inspect training fit and transfer separately

Each training arrangement must individually reach ≥99% accuracy per shape and ≥95%
families correct across all four sizes. Transfer remains descriptive, without accuracy
gates or a milestone-completion claim. Compare identical populations; the old aggregate
across seven transfer arrangements is not the comparator for this four-arrangement split.


In [ ]:
import json

from IPython.display import HTML, FileLink, display

print("Training criteria:", json.dumps(report["training_assessment"], indent=2))
for role, metrics in report["aggregates"].items():
    print(role, {k: metrics["summary"][k] for k in ("total", "accuracy", "loss")})
for name, metrics in report["arrangements"].items():
    print(name, metrics["summary"]["breakdowns"]["shape"])
    keys = ("total", "correct", "fraction")
    print("Correct families:", {k: metrics["families"][k] for k in keys})
comparison = json.loads((run.root / "diagnosis/comparison.json").read_text())
print("Transfer comparison:", json.dumps(comparison["transfer_aggregate"], indent=2))
for name, changes in comparison["matched_changes"].items():
    print(name, {k: v for k, v in changes.items() if k != "details"})
display(HTML((run.root / "diagnosis/inspection.html").read_text()))


## Preserve artifacts

Download `milestone2_multi_arrangement_artifacts.zip` for review. It contains the fresh
checkpoint, manifest, protocol, exposure counts, histories, final predictions, per-arrangement
and aggregate metrics, reference comparisons, preview, hashes and logs. Preserve prior runs.


In [ ]:
archive = archive_multi_arrangement(run)
print("Multi-arrangement archive:", archive)
display(FileLink(str(archive)))
